# EX_01 — Embeddings básicos (ejercicios)

**Notebook de referencia:** `notebook/01_Introduccion_NLP_Embeddings_Basicos.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — One-hot desde vocabulario

Dado un vocabulario fijo y una frase tokenizada en palabras, construye una matriz one-hot `(seq_len, vocab_size)` sin sklearn (solo NumPy).


In [1]:
import numpy as np

# Vocabulario fijo y frase de prueba
vocab = ["cat", "sat", "mat", "the"]
sentence = ["the", "cat", "sat"]  # Frase tokenizada de longitud seq_len = 3

# 1. Crear el diccionario de índices a partir del vocabulario
# Esto mapea: {"cat": 0, "sat": 1, "mat": 2, "the": 3}
vocab_to_index = {word: i for i, word in enumerate(vocab)}

# Definimos las dimensiones de nuestra matriz matemática
seq_len = len(sentence)
vocab_size = len(vocab)

# 2. Inicializar la matriz One-Hot con ceros absolutos
# Tendrá tantas filas como palabras tenga la frase, y tantas columnas como el vocabulario total
one_hot_matrix = np.zeros((seq_len, vocab_size), dtype=np.float32)

# 3. Rellenar con un '1' la posición correspondiente a cada palabra de la frase
for row_idx, word in enumerate(sentence):
    if word in vocab_to_index:
        col_idx = vocab_to_index[word]
        one_hot_matrix[row_idx, col_idx] = 1.0
    else:
        # Nota: Si una palabra no estuviera en el vocabulario, su fila se quedaría a 0 completo (Out-Of-Vocabulary)
        pass

# 4. Mostrar el resultado de forma clara
print("--- DICCIONARIO DE VOCABULARIO ---")
print(vocab_to_index)

print(f"\n--- FRASE DE ENTRADA (Longitud = {seq_len}) ---")
print(sentence)

print(f"\n--- MATRIZ ONE-HOT GENERADA {one_hot_matrix.shape} ---")
print(one_hot_matrix)


--- DICCIONARIO DE VOCABULARIO ---
{'cat': 0, 'sat': 1, 'mat': 2, 'the': 3}

--- FRASE DE ENTRADA (Longitud = 3) ---
['the', 'cat', 'sat']

--- MATRIZ ONE-HOT GENERADA (3, 4) ---
[[0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]]


## Actividad 2 — Similitud coseno

Implementa `cosine_similarity(a, b)` para vectores 1D y compara dos palabras usando sus filas one-hot (esperado: 0 o 1). Luego discute por qué one-hot no captura similitud semántica.


In [2]:
import numpy as np

# Recuperamos los datos del vocabulario de la actividad anterior
vocab = ["cat", "sat", "mat", "the"]
vocab_to_index = {word: i for i, word in enumerate(vocab)}

# 1. Definición de la función matemática de similitud coseno
def cosine_similarity(a, b):
    # La fórmula es: (a . b) / (||a|| * ||b||)
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)

    # Controlamos la división por cero por seguridad matemática
    if norm_a == 0 or norm_b == 0:
        return 0.0

    return dot_product / (norm_a * norm_b)

# 2. Creamos los vectores One-Hot puros para dos palabras del vocabulario: "cat" y "sat"
# Construimos sus filas como vectores 1D de tamaño vocab_size (4)
one_hot_cat = np.zeros(len(vocab))
one_hot_cat[vocab_to_index["cat"]] = 1.0  # Vector: [1, 0, 0, 0]

one_hot_sat = np.zeros(len(vocab))
one_hot_sat[vocab_to_index["sat"]] = 1.0  # Vector: [0, 1, 0, 0]

# Extra: Un vector idéntico a "cat" para comprobar el caso de similitud máxima (1)
one_hot_cat_copia = np.copy(one_hot_cat)

# 3. Calcular las similitudes
sim_diferentes = cosine_similarity(one_hot_cat, one_hot_sat)
sim_identicos  = cosine_similarity(one_hot_cat, one_hot_cat_copia)

print("--- COMPROBACIÓN DE SIMILITUD COSENO ---")
print(f"Vector 'cat': {one_hot_cat}")
print(f"Vector 'sat': {one_hot_sat}")
print(f"Similitud coseno entre 'cat' y 'sat' (esperado 0): {sim_diferentes:.1f}")
print(f"Similitud coseno entre 'cat' y 'cat' (esperado 1): {sim_identicos:.1f}")


--- COMPROBACIÓN DE SIMILITUD COSENO ---
Vector 'cat': [1. 0. 0. 0.]
Vector 'sat': [0. 1. 0. 0.]
Similitud coseno entre 'cat' y 'sat' (esperado 0): 0.0
Similitud coseno entre 'cat' y 'cat' (esperado 1): 1.0


## Actividad 3 — Ventana con Gensim (lectura + entrenamiento mínimo)

Entrena un `Word2Vec` minúsculo sobre `sentences` (lista de listas de tokens) con `vector_size=8`, `window=2`, `min_count=1`. Imprime el vector de una palabra y la similitud entre dos.

*Hint:* `from gensim.models import Word2Vec`.


In [4]:
!pip install gensim
from gensim.models import Word2Vec

# El corpus de texto de juguete proporcionado
sentences = [
    ["the", "cat", "sat", "on", "the", "mat"],
    ["the", "dog", "sat", "on", "the", "log"],
]

# 1. Entrenar el modelo Word2Vec con los parámetros solicitados
model = Word2Vec(
    sentences=sentences,
    vector_size=8,      # Cada palabra se representará con un vector de 8 dimensiones
    window=2,           # Tamaño de la ventana de contexto (2 palabras a la izquierda y 2 a la derecha)
    min_count=1,        # Ignora palabras con frecuencia menor a esta (1 incluye todo el vocabulario)
    workers=1,          # Un solo hilo para garantizar consistencia en pruebas pequeñas
    seed=42             # Fijamos la semilla de aleatoriedad
)

# 2. Acceder a los vectores entrenados (KeyedVectors se guarda en .wv)
word_vectors = model.wv

# 3. Inspeccionar el vector embebido de una palabra (por ejemplo, "cat")
vector_cat = word_vectors["cat"]

# 4. Calcular la similitud semántica por coseno entre dos palabras ("cat" y "dog")
similitud_cat_dog = word_vectors.similarity("cat", "dog")

# 5. Imprimir los resultados por pantalla
print("--- EMBEDDINGS DENSOS CON WORD2VEC ---")
print(f"Vector entrenado para la palabra 'cat' (Tamaño {len(vector_cat)}):")
print(vector_cat)

print("\n--- SIMILITUD SEMÁNTICA CAPTURADA ---")
print(f"Similitud coseno entre 'cat' y 'dog': {similitud_cat_dog:.4f}")

# Extra: comparamos con una palabra funcional que no tiene nada que ver para ver la diferencia
similitud_cat_the = word_vectors.similarity("cat", "the")
print(f"Similitud coseno entre 'cat' y 'the': {similitud_cat_the:.4f}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 65.6 MB/s eta 0:00:00
--- EMBEDDINGS DENSOS CON WORD2VEC ---
Vector entrenado para la palabra 'cat' (Tamaño 8):
[ 0.04447974  0.06959587  0.06497486 -0.07634033 -0.03402349 -0.00831975
 -0.00055212 -0.11404906]

--- SIMILITUD SEMÁNTICA CAPTURADA ---
Similitud coseno entre 'cat' y 'dog': 0.0902
Similitud coseno entre 'cat' y 'the': -0.0541
